In [43]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PowerTransformer, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import joblib

In [44]:
df = pd.read_csv('./../Telco-Customer-Churn.csv')

In [45]:
df.shape

(7043, 21)

In [46]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


# Remove Duplicates

In [47]:
df.drop_duplicates(inplace=True)

In [48]:
df.shape

(7043, 21)

# Type Casting

In [49]:
df['TotalCharges'] = df['TotalCharges'].replace(' ', 0.0)
df['TotalCharges'] = df['TotalCharges'].astype('double')

# Value Corrections

In [50]:
col_vals = {
    'MultipleLines': {
        'old': 'No phone service',
        'new': 'No'
    },
    'OnlineSecurity': {
        'old': 'No internet service',
        'new': 'No'
    },
    'OnlineBackup': {
        'old': 'No internet service',
        'new': 'No'
    },
    'DeviceProtection': {
        'old': 'No internet service',
        'new': 'No'
    },
    'TechSupport': {
        'old': 'No internet service',
        'new': 'No'
    },
    'StreamingTV': {
        'old': 'No internet service',
        'new': 'No'
    },
    'StreamingMovies': {
        'old': 'No internet service',
        'new': 'No'
    },
}

In [51]:
for col in col_vals.keys():
    df[col] = df[col].replace(col_vals[col]['old'], col_vals[col]['new'])

# Define Features & Target

In [52]:
exclude_cols = ['customerID', 'Churn']

X = df.drop(exclude_cols, axis=1)
y = df['Churn']

In [53]:
y = y.map({'Yes':1, 'No': 0})

# Train-Test Split

In [54]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Build Pipelines

In [55]:
cat_yes_no_cols = ['gender', 
                    'Partner', 
                    'Dependents', 
                    'PhoneService', 
                    'MultipleLines', 
                    'OnlineSecurity', 
                    'OnlineBackup', 
                    'DeviceProtection', 
                    'TechSupport', 
                    'StreamingTV', 
                    'StreamingMovies',
                    'PaperlessBilling']
cat_internetservice_cols = ['InternetService']
cat_contract_cols = ['Contract']
cat_paymentmethod_cols = ['PaymentMethod']

cat_yes_no_val_order = [['No', 'Yes']]*len(cat_yes_no_cols)
cat_internetservice_val_order = [['No', 'DSL', 'Fiber optic']]*len(cat_internetservice_cols)
cat_contract_val_order = [['Month-to-month', 'One year', 'Two year']]*len(cat_contract_cols)
cat_paymentmethod_val_order = [['Electronic check', 'Mailed check', 'Bank transfer (automatic)',
 'Credit card (automatic)']]*len(cat_paymentmethod_cols)


num_scale_cols = ['tenure', 'MonthlyCharges']
num_transform_cols = ['TotalCharges']

In [56]:
yesno_encoder_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
        ('onehot_yesno_encode', OneHotEncoder(categories=cat_yes_no_val_order, drop='first', sparse_output=False, handle_unknown='ignore')),
])

internetservice_encoder_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
        ('onehot_internetservice_encode', OneHotEncoder(categories=cat_internetservice_val_order, drop='first', sparse_output=False, handle_unknown='ignore')),
])

paymentmethod_encoder_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
        ('onehot_paymentmethod_encode', OneHotEncoder(categories=cat_paymentmethod_val_order, drop='first', sparse_output=False, handle_unknown='ignore')),
])

contract_encoder_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
        ('ordinal_contract_encode', OrdinalEncoder(categories=cat_contract_val_order))
])

In [57]:
scalling_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=-1, add_indicator=True)),
    ('scalling', StandardScaler())
])

transform_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=-1, add_indicator=True)),
    ('transform', PowerTransformer())
])


In [58]:
preprocessor = ColumnTransformer([
    ('ohe_yesno_encoder', yesno_encoder_pipeline, cat_yes_no_cols),
    ('ohe_internetservice_encoder', internetservice_encoder_pipeline, cat_internetservice_cols),
    ('ohe_paymentmethod_encoder', paymentmethod_encoder_pipeline, cat_paymentmethod_cols),
    ('ordinal_contract_encoder', contract_encoder_pipeline, cat_contract_cols),
    ('scalling', scalling_pipeline, num_scale_cols),
    ('transforming', transform_pipeline, num_transform_cols)
], remainder='passthrough')

# Fit & Transform

In [59]:
X_train_processed = preprocessor.fit_transform(X_train)  
X_test_processed  = preprocessor.transform(X_test)

C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [60]:
feature_names = preprocessor.get_feature_names_out()
print(feature_names)

['ohe_yesno_encoder__gender_Yes' 'ohe_yesno_encoder__Partner_Yes'
 'ohe_yesno_encoder__Dependents_Yes' 'ohe_yesno_encoder__PhoneService_Yes'
 'ohe_yesno_encoder__MultipleLines_Yes'
 'ohe_yesno_encoder__OnlineSecurity_Yes'
 'ohe_yesno_encoder__OnlineBackup_Yes'
 'ohe_yesno_encoder__DeviceProtection_Yes'
 'ohe_yesno_encoder__TechSupport_Yes' 'ohe_yesno_encoder__StreamingTV_Yes'
 'ohe_yesno_encoder__StreamingMovies_Yes'
 'ohe_yesno_encoder__PaperlessBilling_Yes'
 'ohe_internetservice_encoder__InternetService_DSL'
 'ohe_internetservice_encoder__InternetService_Fiber optic'
 'ohe_paymentmethod_encoder__PaymentMethod_Mailed check'
 'ohe_paymentmethod_encoder__PaymentMethod_Bank transfer (automatic)'
 'ohe_paymentmethod_encoder__PaymentMethod_Credit card (automatic)'
 'ordinal_contract_encoder__Contract' 'scalling__tenure'
 'scalling__MonthlyCharges' 'transforming__TotalCharges'
 'remainder__SeniorCitizen']


# Save Preprocessor & Splits

In [61]:
joblib.dump(preprocessor, 'preprocessor.pkl')

['preprocessor.pkl']

In [62]:
np.save('X_train.npy', X_train_processed)
np.save('X_test.npy',  X_test_processed)
np.save('y_train.npy', y_train.values)
np.save('y_test.npy',  y_test.values)

In [63]:
print("Preprocessing pipeline complete!")
print(f"X_train shape: {X_train_processed.shape}")
print(f"X_test shape:  {X_test_processed.shape}")

Preprocessing pipeline complete!
X_train shape: (5634, 22)
X_test shape:  (1409, 22)
